# Example: An MCP Server for the National Weather Service API
In this example, we turn our software development kit (SDK) for the National Weather Service (NWS) API into a Model Context Protocol (MCP) server. The server offers two tools to an MCP host: one finds a location, and one downloads the hourly forecast. We play the host, and make the calls that a host would make for its language model.

> __Learning Objectives:__
>
> By the end of this example, you should be able to:
>
> * __Start and discover an MCP server:__ A host starts a stdio server as a subprocess, and sends it one line of JSON text per message. We'll start the weather server from Julia, and read the messages that discover the server.
> * __List and call the tools of a server:__ Each tool has a name, a description, and an input schema. We'll list the two tools of the weather server, and call each tool with arguments that match its schema.
> * __Tell a protocol error from a tool execution error:__ A wrong request gets a JSON-RPC error, while a tool that fails returns a normal result with its error flag set. We'll trigger both kinds of errors, and read what a host would give to its model.

Let's get started!
___

## Setup, Data, and Prerequisites

First, we load the local [`Include.jl`](Include.jl) setup file and its required packages.

> __Include:__ The [`include(...)` command](https://docs.julialang.org/en/v1/base/base/#include) evaluates [`Include.jl`](Include.jl) in the notebook's global scope. The setup file activates the course project, defines local paths, and loads the course package, this example's code in the [src](src) folder, and the packages used here.

Let's set up our code environment:

In [1]:
# Load packages and paths from this notebook's local setup file -
include(joinpath(@__DIR__, "Include.jl"));

In addition to standard Julia libraries, we'll use [the `VLDataScienceMachineLearningPackage.jl` package](https://github.com/varnerlab/VLDataScienceMachineLearningPackage.jl). Check out [the documentation](https://varnerlab.github.io/VLDataScienceMachineLearningPackage.jl/dev/) for more information on the functions, types, and data used in this material.

The code for this example is in the [src](src) folder. The first four files are our NWS SDK:

* [src/Types.jl](src/Types.jl) holds the endpoint models, and the models of an MCP tool, server, and client.
* [src/Factory.jl](src/Factory.jl) holds the factory methods that build the request URLs, the tools, and the server.
* [src/Network.jl](src/Network.jl) holds the functions that make the HTTP calls.
* [src/Handler.jl](src/Handler.jl) holds the handlers, the functions that process a response.
* [src/Tools.jl](src/Tools.jl) holds the two tools of the weather server.
* [src/Server.jl](src/Server.jl) holds the server loop: read a message, process it, and write the response.
* [src/Client.jl](src/Client.jl) holds the client functions that our notebook uses to talk to the server.

The [server.jl](server.jl) file is the program that a host starts. It loads the code in the [src](src) folder, builds the weather server, and runs the server loop.

### Constants
We'll set the location of Ithaca, NY, and the command that starts the weather server. The command runs [server.jl](server.jl) in the course environment, with the Julia program that runs this notebook:

In [2]:
latitude = 42.443961; # latitude of Ithaca, NY (units: decimal degrees, north is positive)
longitude = -76.501881; # longitude of Ithaca, NY (units: decimal degrees, east is positive)
julia_program = joinpath(Sys.BINDIR, "julia"); # the Julia program that runs this notebook
server_command = `$(julia_program) --startup-file=no --project=$(CHEME5800_ROOT) $(CHEME5800_L13C_SERVER)`; # the command that starts the weather server

___
## Task 1: Start the Server and Discover It
In this task, we start the weather server as a subprocess, and ask the server who it is.

[The `connect(...)` function](src/Client.jl) runs the server command, and connects to the standard input and the standard output of the server. It returns [a `MyMCPClientModel` instance](src/Types.jl). We pass `verbose = true`, so the client prints every message that it sends and receives.

Let's start the server:

In [3]:
client = connect(server_command; verbose = true); # start the server, and connect to it

Next, we send a `server/discover` request with [the `discover(...)` function](src/Client.jl). The server loads its packages before it reads its first message, so the first response takes a few seconds.

> __What is happening behind the scenes?__ [The `send_request(...)` function](src/Client.jl) adds the `_meta` object to the parameters of the request. It writes the request as one line of JSON text to the standard input of the server. It then reads one line from the standard output of the server, and parses it.

We store the response in the `discover_response` variable:

In [4]:
discover_response = discover(client); # send a server/discover request, and read the response

Client → Server


{
  "id": 1,
  "jsonrpc": "2.0",
  "method": "server/discover",
  "params": {
    "_meta": {
      "io.modelcontextprotocol/clientCapabilities": {},
      "io.modelcontextprotocol/clientInfo": {
        "name": "cheme-5800-notebook",
        "version": "1.0.0"
      },
      "io.modelcontextprotocol/protocolVersion": "2026-07-28"
    }
  }
}


Server → Client
{
  "id": 1,
  "jsonrpc": "2.0",
  "result": {
    "_meta": {
      "io.modelcontextprotocol/serverInfo": {
        "name": "cheme-5800-weather",
        "version": "1.0.0"
      }
    },
    "cacheScope": "public",
    "capabilities": {
      "tools": {}
    },
    "instructions": "Weather data for locations in the United States from the National Weather Service. Give each location as a latitude and a longitude in decimal degrees.",
    "resultType": "complete",
    "supportedVersions": [
      "2026-07-28"
    ],
    "ttlMs": 3600000
  }
}


__What do we see?__ The request names protocol version `2026-07-28` in its `_meta` object. The response has the same `id` as the request:

* The `supportedVersions` field says that the server speaks protocol version `2026-07-28`.
* The `capabilities` field says that the server offers tools.
* The `_meta` object names the server, `cheme-5800-weather`, and its version.

Each later message is longer, so we turn off the printing of every message:

In [5]:
client.verbose = false; # do not print every message from here on

___
## Task 2: List the Tools
In this task, we ask the server for its tools, and read the input schema of one tool.

Let's send a `tools/list` request with [the `listtools(...)` function](src/Client.jl), and put the name, the required arguments, and the description of each tool into a table:

In [6]:
list_response = listtools(client); # send a tools/list request, and read the response
tools = list_response["result"]["tools"]; # one dictionary for each tool

# one row for each tool -
tool_table = DataFrame(name = String[], required = String[], description = String[]);
for tool ∈ tools
    required_arguments = join(tool["inputSchema"]["required"], ", "); # the arguments that every call must give
    push!(tool_table, (tool["name"], required_arguments, tool["description"]));
end
pretty_table(tool_table; backend = :text, display_size = (-1, -1),
    table_format = TextTableFormat(borders = text_table_borders__compact));

 --------------------- --------------------- ---------------------------------------------------------------------------------------------------------------------------------------------------------------
                 name              required                                                                                                                                                     description 
               String                String                                                                                                                                                          String 
 --------------------- --------------------- ---------------------------------------------------------------------------------------------------------------------------------------------------------------
  get_hourly_forecast   latitude, longitude   Get the hourly weather forecast for a location in the United States from the National Weather Service: temperature, wind, and a short 

The server has two tools, and both of them need a latitude and a longitude. A language model reads the description of each tool to decide which tool fits a question.

Next, let's find the hourly forecast tool in the list, and print its input schema:

In [7]:
forecast_tool = nothing;
for tool ∈ tools
    if (tool["name"] == "get_hourly_forecast")
        forecast_tool = tool;
    end
end
println(JSON.json(forecast_tool["inputSchema"]; pretty = true)) # the JSON Schema of the arguments

{
  "properties": {
    "hours": {
      "description": "Number of hours to return, starting now (default: 12)",
      "maximum": 48,
      "minimum": 1,
      "type": "integer"
    },
    "latitude": {
      "description": "Latitude in decimal degrees, north is positive, e.g., 42.444 for Ithaca, NY",
      "maximum": 90,
      "minimum": -90,
      "type": "number"
    },
    "longitude": {
      "description": "Longitude in decimal degrees, east is positive, e.g., -76.502 for Ithaca, NY",
      "maximum": 180,
      "minimum": -180,
      "type": "number"
    }
  },
  "required": [
    "latitude",
    "longitude"
  ],
  "type": "object"
}


__What do we see?__ The schema describes a JSON object with three properties. The `latitude` and `longitude` are numbers, and both are required. The `hours` argument is an optional integer between `1` and `48`.

___
## Task 3: Call the Tools
In this task, we call each tool as a host would for its language model. Then, we trigger both kinds of errors.

Let's call the `get_location` tool for Ithaca, NY with [the `calltool(...)` function](src/Client.jl), and print the response of the server:

In [8]:
location_arguments = Dict{String,Any}("latitude" => latitude, "longitude" => longitude); # the arguments of the call
location_response = calltool(client, "get_location", location_arguments);
println(JSON.json(location_response; pretty = true)) # the response of the server

{
  "id": 3,
  "jsonrpc": "2.0",
  "result": {
    "_meta": {
      "io.modelcontextprotocol/serverInfo": {
        "name": "cheme-5800-weather",
        "version": "1.0.0"
      }
    },
    "content": [
      {
        "text": "{\"city\":\"Ithaca\",\"forecast_office\":\"BGM\",\"grid_x\":44,\"grid_y\":70,\"state\":\"NY\",\"time_zone\":\"America/New_York\"}",
        "type": "text"
      }
    ],
    "isError": false,
    "resultType": "complete",
    "structuredContent": {
      "city": "Ithaca",
      "forecast_office": "BGM",
      "grid_x": 44,
      "grid_y": 70,
      "state": "NY",
      "time_zone": "America/New_York"
    }
  }
}


__What do we see?__ The `isError` flag of the result is `false`, so the tool ran without an error:

* The `structuredContent` object holds the result as JSON data. The NWS forecast office for Ithaca, NY is `BGM`, the office in Binghamton, NY.
* The `content` array holds the same result as text. A host gives this text to its language model.

Now, let's ask for the forecast of the next six hours. We put the `periods` of the result into a table:

In [9]:
forecast_arguments = Dict{String,Any}("latitude" => latitude, "longitude" => longitude, "hours" => 6);
forecast_response = calltool(client, "get_hourly_forecast", forecast_arguments);
forecast_result = forecast_response["result"]["structuredContent"]; # the result as JSON data

# one row for each hour of the forecast -
forecast_table = DataFrame(start_time = String[], temperature = Int64[], wind = String[], short_forecast = String[]);
for period ∈ forecast_result["periods"]
    wind = period["wind_speed"] * " " * period["wind_direction"]; # e.g., 10 mph NW
    push!(forecast_table, (period["start_time"], period["temperature"], wind, period["short_forecast"]));
end
pretty_table(forecast_table; backend = :text, display_size = (-1, -1),
    table_format = TextTableFormat(borders = text_table_borders__compact));

 --------------------------- ------------- ---------- ----------------------
                 start_time   temperature       wind         short_forecast 
                     String         Int64     String                 String 
 --------------------------- ------------- ---------- ----------------------
  2026-11-12T22:00:00-05:00            37    7 mph N             Light Rain
  2026-11-12T23:00:00-05:00            36    8 mph N          Rain And Snow
  2026-11-13T00:00:00-05:00            35    7 mph N          Rain And Snow
  2026-11-13T01:00:00-05:00            34   7 mph NW          Rain And Snow
  2026-11-13T02:00:00-05:00            33   8 mph NW   Rain And Snow Likely
  2026-11-13T03:00:00-05:00            34   8 mph NW      Light Snow Likely
 --------------------------- ------------- ---------- ----------------------


The temperatures are in degrees Fahrenheit. A host would give the text of this result to its model, and the model would use it to answer a question such as "Will it rain in Ithaca tonight?".

### Two Kinds of Errors
First, let's call a tool that the server does not have:

In [10]:
unknown_response = calltool(client, "get_snowfall", location_arguments); # the server has no get_snowfall tool
println(JSON.json(unknown_response; pretty = true))

{
  "error": {
    "code": -32602,
    "message": "Unknown tool: get_snowfall"
  },
  "id": 5,
  "jsonrpc": "2.0"
}


__What do we see?__ The response holds an `error` object, and no `result`. The code `-32602` means invalid params: the request named a tool that the server does not have. This is a protocol error, which usually points to a bug in the host.

Next, let's ask for the forecast in Paris, France. The request is fine, but the NWS covers only the United States:

In [11]:
paris_arguments = Dict{String,Any}("latitude" => 48.8566, "longitude" => 2.3522); # Paris, France
paris_response = calltool(client, "get_hourly_forecast", paris_arguments);
println(JSON.json(paris_response; pretty = true))

{
  "id": 6,
  "jsonrpc": "2.0",
  "result": {
    "_meta": {
      "io.modelcontextprotocol/serverInfo": {
        "name": "cheme-5800-weather",
        "version": "1.0.0"
      }
    },
    "content": [
      {
        "text": "ArgumentError: HTTP GET call failed with status code 404",
        "type": "text"
      }
    ],
    "isError": true,
    "resultType": "complete"
  }
}


__What do we see?__ The response holds a normal `result`, with `isError` set to `true`. The tool ran, and the NWS answered with status code `404`, i.e., not found. This is a tool execution error, so a host gives the text to its model, and the model can tell the user that the location is outside the United States.

### Close the Connection
We are done with the server. [The `close(...)` function](src/Client.jl) closes the standard input of the server. The server loop reaches the end of its input, and the server exits:

In [12]:
close(client); # close the standard input of the server, and wait for the server to exit

> __Connect a host to the weather server:__ A host such as Claude Code starts the same server with the same command. For example, the command below registers the weather server with Claude Code, where `/path/to/L13c/server.jl` is the path of the [server.jl](server.jl) file on your computer:
>
> `claude mcp add weather -- julia --startup-file=no /path/to/L13c/server.jl`
>
> Claude Code then lists the tools of the server. It asks for your approval before its model calls a tool, unless your permission settings already allow the call.

### Check: Weather server
Let's check the responses of the server: the discover response, the tools, the forecast, and both errors:

In [13]:
@testset "Weather server" begin
    tool_names = String[];
    for tool ∈ tools
        push!(tool_names, tool["name"]);
    end
    @test "2026-07-28" ∈ discover_response["result"]["supportedVersions"] # the server speaks our protocol version
    @test sort(tool_names) == ["get_hourly_forecast", "get_location"] # the server offers two tools
    @test location_response["result"]["isError"] == false # the location call ran without an error
    @test forecast_response["result"]["isError"] == false # the forecast call ran without an error
    @test length(forecast_result["periods"]) == 6 # we asked for six hours
    @test unknown_response["error"]["code"] == -32602 # an unknown tool is a protocol error
    @test paris_response["result"]["isError"] == true # a location outside the United States is a tool execution error
    @test process_exited(client.process) == true # the server exited when we closed the connection
end;

Test Summary:  | Pass  

Total  Time
Weather server |    8      8  0.3s


___
## Summary
In this example, we started an MCP server that wraps the NWS SDK, called its two tools as a host would, and triggered both kinds of errors.

> __Key Takeaways:__
>
> * __A server is a program that the host starts:__ We started the weather server as a subprocess, and sent it one line of JSON text per message. The server answered a discover request with its protocol versions, its capabilities, and its name.
> * __Tools wrap the SDK:__ Each tool checks its arguments, calls the NWS API through the SDK, and returns a small result with plain field names. The result comes back both as text for a language model and as JSON data for a program.
> * __Two kinds of errors:__ A call to a tool that the server does not have got a JSON-RPC error, with the code for invalid params. A forecast call for Paris, France got a normal result with its error flag set, because the NWS covers only the United States.

Any MCP host that supports protocol version `2026-07-28` can now use our weather server, with no code written for that host.
___